In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from statsbombpy import sb
from sklearn.preprocessing import RobustScaler
import warnings

pd.set_option("future.no_silent_downcasting", True)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

In [ ]:
def get_all_matches():
    print("Fetching competitions...")
    comps = sb.competitions()
    all_matches = []

    print(f"Found {len(comps)} competition-season combinations.")
    for _, row in tqdm(comps.iterrows(), total=len(comps), desc="Fetching matches"):
        matches = sb.matches(
            competition_id=row["competition_id"], season_id=row["season_id"]
        )
        matches["competition_name"] = row["competition_name"]
        matches["season_name"] = row["season_name"]
        all_matches.append(matches)

    return pd.concat(all_matches, ignore_index=True)


df_matches = get_all_matches()
print(f"Total matches found: {len(df_matches)}")

In [ ]:
# HELPERS

def _safe_bool_sum(series):
    """
    Replaces .fillna(False).sum() on object/bool columns without triggering
    """
    return series.notna().astype(bool) & series.astype(object).ne(False)


def _bool_col_sum(df, col):
    """Sum a boolean-flag column safely, returning 0 if column is absent."""
    if col not in df.columns:
        return 0
    return df[col].infer_objects(copy=False).fillna(False).astype(bool).sum()


def _safe_col(df, col, default=0):
    """Return df[col] if it exists, else a Series of `default`."""
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)

In [ ]:
def process_match_data(match_ids, sample_size=None):
    """
    Extracts every available per-player, per-match stat from StatsBomb events.
    """
    if sample_size:
        match_ids = match_ids[:sample_size]

    all_player_stats = []

    for m_id in tqdm(match_ids, desc="Processing matches"):
        try:
            events = sb.events(match_id=m_id)
            df = events.dropna(subset=["player"]).copy()

            # Lineup bio
            lineup_bio = {}
            try:
                raw_lineups = sb.lineups(match_id=m_id)
                for team_name, team_df in raw_lineups.items():
                    for _, row in team_df.iterrows():
                        lineup_bio[row["player_id"]] = {
                            "birth_date": row.get("birth_date", np.nan),
                            "height": row.get("player_height", np.nan),
                            "weight": row.get("player_weight", np.nan),
                            "jersey_number": row.get("jersey_number", np.nan),
                            "nationality": row.get("country", np.nan),
                        }
            except Exception:
                pass

            for (p_id, p_name), p_events in df.groupby(["player_id", "player"]):

                stats = {
                    "player_id": p_id,
                    "player": p_name,
                    "match_id": m_id,
                    "team": p_events["team"].iloc[0],
                    "position": p_events["position"].iloc[0],
                }

                bio = lineup_bio.get(p_id, {})
                stats["birth_date"] = bio.get("birth_date", np.nan)
                stats["height"] = bio.get("height", np.nan)
                stats["weight"] = bio.get("weight", np.nan)
                stats["jersey_number"] = bio.get("jersey_number", np.nan)
                stats["nationality"] = bio.get("nationality", np.nan)

                # Minutes played
                stats["minutes_played"] = min(int(p_events["minute"].max()) + 1, 90)

                # Touches & pressure
                stats["touches"] = p_events["location"].notna().sum()
                stats["under_pressure_actions"] = _bool_col_sum(
                    p_events, "under_pressure"
                )

                # PASSING
                passes = p_events[p_events["type"] == "Pass"]
                stats["total_passes"] = len(passes)

                if not passes.empty:
                    stats["pass_completion"] = passes["pass_outcome"].isna().mean()
                    stats["avg_pass_length"] = (
                        passes["pass_length"].mean()
                        if "pass_length" in passes.columns
                        else 0
                    )
                    stats["short_passes"] = (
                        (passes["pass_length"] < 15).sum()
                        if "pass_length" in passes.columns
                        else 0
                    )
                    stats["medium_passes"] = (
                        passes["pass_length"].between(15, 30).sum()
                        if "pass_length" in passes.columns
                        else 0
                    )
                    stats["long_passes"] = (
                        (passes["pass_length"] > 30).sum()
                        if "pass_length" in passes.columns
                        else 0
                    )
                    stats["crosses"] = _bool_col_sum(passes, "pass_cross")
                    stats["through_balls"] = _bool_col_sum(passes, "pass_through_ball")
                    stats["switches"] = _bool_col_sum(passes, "pass_switch")
                    stats["cut_backs"] = _bool_col_sum(passes, "pass_cut_back")
                    stats["inswinging_crosses"] = _bool_col_sum(
                        passes, "pass_inswinging"
                    )
                    stats["outswinging_crosses"] = _bool_col_sum(
                        passes, "pass_outswinging"
                    )
                    stats["under_pressure_passes"] = _bool_col_sum(
                        passes, "under_pressure"
                    )
                    stats["key_passes"] = _bool_col_sum(passes, "pass_shot_assist")
                    stats["goal_assists"] = _bool_col_sum(passes, "pass_goal_assist")

                    # Passes into the penalty box (end location x≥102, 18≤y≤62)
                    if "pass_end_location" in passes.columns:
                        stats["passes_into_box"] = (
                            passes["pass_end_location"]
                            .apply(
                                lambda loc: isinstance(loc, list)
                                and len(loc) == 2
                                and loc[0] >= 102
                                and 18 <= loc[1] <= 62
                            )
                            .sum()
                        )
                    else:
                        stats["passes_into_box"] = 0

                    # Set-piece subtypes
                    pass_type = _safe_col(passes, "pass_type", default=pd.NA)
                    stats["corner_passes"] = (pass_type == "Corner").sum()
                    stats["free_kick_passes"] = (pass_type == "Free Kick").sum()
                    stats["throw_in_passes"] = (pass_type == "Throw-in").sum()
                    stats["goal_kick_passes"] = (pass_type == "Goal Kick").sum()

                    # Progressive passes (end_x - start_x > 10)
                    if (
                        "location" in passes.columns
                        and "pass_end_location" in passes.columns
                    ):
                        start_x = passes["location"].str[0]
                        end_x = passes["pass_end_location"].str[0]
                        stats["progressive_passes"] = ((end_x - start_x) > 10).sum()
                    else:
                        stats["progressive_passes"] = 0
                else:
                    stats.update(
                        {
                            "pass_completion": 0,
                            "avg_pass_length": 0,
                            "short_passes": 0,
                            "medium_passes": 0,
                            "long_passes": 0,
                            "crosses": 0,
                            "through_balls": 0,
                            "switches": 0,
                            "cut_backs": 0,
                            "inswinging_crosses": 0,
                            "outswinging_crosses": 0,
                            "under_pressure_passes": 0,
                            "key_passes": 0,
                            "goal_assists": 0,
                            "passes_into_box": 0,
                            "corner_passes": 0,
                            "free_kick_passes": 0,
                            "throw_in_passes": 0,
                            "goal_kick_passes": 0,
                            "progressive_passes": 0,
                        }
                    )

                # CARRYING
                carries = p_events[p_events["type"] == "Carry"]
                stats["total_carries"] = len(carries)

                if (
                    not carries.empty
                    and "location" in carries.columns
                    and "carry_end_location" in carries.columns
                ):
                    sx = carries["location"].str[0]
                    sy = carries["location"].str[1]
                    ex = carries["carry_end_location"].str[0]
                    ey = carries["carry_end_location"].str[1]
                    dist = np.sqrt((ex - sx) ** 2 + (ey - sy) ** 2)
                    stats["avg_carry_dist"] = dist.mean()
                    stats["total_carry_dist"] = dist.sum()
                    stats["progressive_carries"] = ((ex - sx) > 10).sum()
                    stats["carries_into_box"] = (
                        carries["carry_end_location"]
                        .apply(
                            lambda loc: isinstance(loc, list)
                            and len(loc) == 2
                            and loc[0] >= 102
                            and 18 <= loc[1] <= 62
                        )
                        .sum()
                    )
                else:
                    stats.update(
                        {
                            "avg_carry_dist": 0,
                            "total_carry_dist": 0,
                            "progressive_carries": 0,
                            "carries_into_box": 0,
                        }
                    )

                # SHOOTING
                shots = p_events[p_events["type"] == "Shot"]
                stats["total_shots"] = len(shots)

                if not shots.empty:
                    outcome = _safe_col(shots, "shot_outcome", default=pd.NA)
                    stats["shots_on_target"] = outcome.isin(["Goal", "Saved"]).sum()
                    stats["goals"] = (outcome == "Goal").sum()
                    stats["shots_blocked"] = (outcome == "Blocked").sum()
                    stats["total_xg"] = _safe_col(
                        shots, "shot_statsbomb_xg", default=0
                    ).sum()
                    stats["avg_xg_per_shot"] = _safe_col(
                        shots, "shot_statsbomb_xg", default=0
                    ).mean()
                    stats["shots_first_time"] = _bool_col_sum(shots, "shot_first_time")
                    stats["shots_one_on_one"] = _bool_col_sum(shots, "shot_one_on_one")
                    stats["shots_open_goal"] = _bool_col_sum(shots, "shot_open_goal")
                    stats["headed_shots"] = (
                        _safe_col(shots, "shot_body_part", default=pd.NA) == "Head"
                    ).sum()
                    if "location" in shots.columns:
                        loc = shots["location"]
                        stats["avg_shot_distance"] = np.sqrt(
                            (120 - loc.str[0]) ** 2 + (40 - loc.str[1]) ** 2
                        ).mean()
                    else:
                        stats["avg_shot_distance"] = 0
                else:
                    stats.update(
                        {
                            "shots_on_target": 0,
                            "goals": 0,
                            "shots_blocked": 0,
                            "total_xg": 0,
                            "avg_xg_per_shot": 0,
                            "shots_first_time": 0,
                            "shots_one_on_one": 0,
                            "shots_open_goal": 0,
                            "headed_shots": 0,
                            "avg_shot_distance": 0,
                        }
                    )

                # DEFENSE
                def_types = [
                    "Tackle",
                    "Interception",
                    "Pressure",
                    "Block",
                    "Ball Recovery",
                    "Clearance",
                ]
                def_actions = p_events[p_events["type"].isin(def_types)]
                stats["total_defensive_actions"] = len(def_actions)

                if not def_actions.empty:
                    stats["defensive_height"] = (
                        def_actions["location"].str[0].mean()
                        if "location" in def_actions.columns
                        else np.nan
                    )
                    stats["pressures"] = (def_actions["type"] == "Pressure").sum()
                    stats["tackles"] = (def_actions["type"] == "Tackle").sum()
                    stats["interceptions"] = (
                        def_actions["type"] == "Interception"
                    ).sum()
                    stats["clearances"] = (def_actions["type"] == "Clearance").sum()
                    stats["ball_recoveries"] = (
                        def_actions["type"] == "Ball Recovery"
                    ).sum()
                    blocks = def_actions[def_actions["type"] == "Block"]
                    stats["blocks"] = len(blocks)
                    stats["block_deflections"] = _bool_col_sum(
                        blocks, "block_deflection"
                    )
                    stats["block_offensive"] = _bool_col_sum(blocks, "block_offensive")
                    stats["block_save_blocks"] = _bool_col_sum(
                        blocks, "block_save_block"
                    )
                else:
                    stats.update(
                        {
                            "defensive_height": np.nan,
                            "pressures": 0,
                            "tackles": 0,
                            "interceptions": 0,
                            "clearances": 0,
                            "ball_recoveries": 0,
                            "blocks": 0,
                            "block_deflections": 0,
                            "block_offensive": 0,
                            "block_save_blocks": 0,
                        }
                    )

                stats["counterpresses"] = _bool_col_sum(p_events, "counterpress")

                # DRIBBLES
                dribbles = p_events[p_events["type"] == "Dribble"]
                stats["total_dribbles"] = len(dribbles)
                stats["successful_dribbles"] = (
                    _safe_col(dribbles, "dribble_outcome", default=pd.NA) == "Complete"
                ).sum()
                stats["nutmegs"] = _bool_col_sum(dribbles, "dribble_nutmeg")
                stats["dribble_no_touch"] = _bool_col_sum(dribbles, "dribble_no_touch")

                # DUELS
                duels = p_events[p_events["type"] == "Duel"]
                stats["total_duels"] = len(duels)
                if not duels.empty:
                    won_outcomes = ["Won", "Success In Play", "Success Out"]
                    duel_outcome = _safe_col(duels, "duel_outcome", default=pd.NA)
                    duel_type = _safe_col(duels, "duel_type", default=pd.NA)
                    stats["duels_won"] = duel_outcome.isin(won_outcomes).sum()
                    stats["aerial_duels"] = (duel_type == "Aerial Lost").sum()
                    stats["ground_duels"] = (duel_type == "Tackle").sum()
                else:
                    stats.update({"duels_won": 0, "aerial_duels": 0, "ground_duels": 0})

                # 50/50s
                fifties = p_events[p_events["type"] == "50/50"]
                stats["total_50_50s"] = len(fifties)
                if not fifties.empty:
                    won50 = ["Won", "Success To Team", "Success To Opposition"]
                    stats["50_50s_won"] = (
                        _safe_col(fifties, "50_50_outcome", default=pd.NA)
                        .isin(won50)
                        .sum()
                    )
                else:
                    stats["50_50s_won"] = 0

                # BALL RECEIPTS
                receipts = p_events[p_events["type"] == "Ball Receipt*"]
                stats["ball_receipts"] = len(receipts)
                stats["ball_receipts_incomplete"] = (
                    _safe_col(receipts, "ball_receipt_outcome", default=pd.NA)
                    == "Incomplete"
                ).sum()

                # GOALKEEPER
                gk = p_events[p_events["type"] == "Goalkeeper"]
                stats["gk_total_actions"] = len(gk)
                if not gk.empty:
                    gk_outcome = _safe_col(gk, "goalkeeper_outcome", default=pd.NA)
                    gk_type = _safe_col(gk, "goalkeeper_type", default=pd.NA)
                    stats["gk_saves"] = gk_outcome.isin(
                        ["Saved", "Saved to Post", "No Touch"]
                    ).sum()
                    stats["gk_claims"] = gk_outcome.isin(["Claim", "Success"]).sum()
                    stats["gk_punches"] = _bool_col_sum(gk, "goalkeeper_punched_out")
                    stats["gk_shot_faced"] = gk_type.isin(
                        ["Shot Saved", "Shot Faced"]
                    ).sum()
                else:
                    stats.update(
                        {
                            "gk_saves": 0,
                            "gk_claims": 0,
                            "gk_punches": 0,
                            "gk_shot_faced": 0,
                        }
                    )

                # DISCIPLINE & TURNOVERS
                fc = p_events[p_events["type"] == "Foul Committed"]
                stats["fouls_committed"] = len(fc)
                fc_card = _safe_col(fc, "foul_committed_card", default=pd.NA)
                stats["yellow_cards"] = (fc_card == "Yellow Card").sum()
                stats["red_cards"] = fc_card.isin(["Red Card", "Second Yellow"]).sum()
                stats["penalties_conceded"] = _bool_col_sum(
                    fc, "foul_committed_penalty"
                )

                fw = p_events[p_events["type"] == "Foul Won"]
                stats["fouls_won"] = len(fw)
                stats["penalties_won"] = _bool_col_sum(fw, "foul_won_penalty")

                stats["miscontrols"] = (p_events["type"] == "Miscontrol").sum()
                stats["dispossessed"] = (p_events["type"] == "Dispossessed").sum()
                stats["turnovers"] = stats["miscontrols"] + stats["dispossessed"]

                all_player_stats.append(stats)

        except Exception as e:
            print(f"Error processing match {m_id}: {e}")
            continue

    if not all_player_stats:
        return pd.DataFrame()

    return pd.DataFrame(all_player_stats)


player_match_logs = process_match_data(df_matches["match_id"].tolist())

In [ ]:
player_match_logs.to_csv("profiles.csv", index=False)